<a href="https://colab.research.google.com/github/nilum2002/Fine-Tune-LLMs-/blob/Main/gemma2-instruct-sinfintune_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U bitsandbytes==0.49.0
!pip install -q -U peft==0.18.0
!pip install -q -U trl==0.26.2
!pip install -q -U accelerate
!pip install -q -U datasets==4.4.2
!pip install -q -U transformers==4.57.3

In [ ]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM # for generating some text based on decoder based transformer
from transformers import BitsAndBytesConfig, GemmaTokenizer


In [ ]:
from google.colab import userdata
import os


os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Thimira/sinhala-llm-dataset-llama-prompt-format/" + splits["train"])

In [ ]:
model_id = "google/gemma-2b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # all 32 bit weights converts in to 4 bits
    bnb_4bit_quant_type="nf4", # 4-bit normal Float
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token = os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,
                                             token = os.environ["HF_TOKEN"],
                                             device_map={"":0}
)


In [ ]:

lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM"
)

In [ ]:
def split_text(text):
    # Assuming the format is always '<s>[INST] prompt [/INST] response</s>'
    parts = text.split('[/INST]', 1)
    if len(parts) == 2:
        prompt_part = parts[0]
        response_part = parts[1]

        # Extracting the actual prompt after '<s>[INST]'
        prompt = prompt_part.replace('<s>[INST]', '').strip()

        # Extracting the actual response before '</s>'
        response = response_part.replace('</s>', '').strip()
        return prompt, response
    else:
        # Handle cases where the format might not match, or return original text
        return '', text # Or raise an error, or log a warning

df[['prompt', 'response']] = df['text'].apply(lambda x: pd.Series(split_text(x)))
df = df.drop(columns=['text'])
print(df.head())

In [ ]:
from datasets import DatasetDict, Dataset

dataset = Dataset.from_pandas(df)

splitted_dataset = dataset.train_test_split(test_size=0.1, seed=42)

data = DatasetDict({
    "train": splitted_dataset["train"],
    "test": splitted_dataset["test"]
})

print(data)

In [ ]:
def formatting_func_str(example):
    return f"<s>[INST] {example['prompt']} [/INST] {example['response']}</s>"

# Pre-process the dataset to create a 'text' column
data["train"] = data["train"].map(lambda example: {"text": formatting_func_str(example)}, remove_columns=["prompt", "response"])

trainer = SFTTrainer(
    model = model,
    train_dataset = data["train"],
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 1000,
        learning_rate = 2e-4,
        fp16 = False,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "paged_adamw_8bit"
    ),
    peft_config = lora_config
    # The formatting_func is no longer needed here as the dataset is already pre-formatted to 'text'
    # SFTTrainer will look for the 'text' column by default.
)

**Reasoning**:
The `SFTTrainer` has been successfully initialized in the previous step. The next logical action is to start the fine-tuning process by calling the `train()` method on the `trainer` object.



In [ ]:
trainer.train()

In [ ]:
# model_name = "gemma-2b-it-sinhala-finetune"
# trainer.push_to_hub(f"Nilum/{model_name}", private=True)

In [ ]:
# model_name = "gemma-2b-it-sinhala-finetune"
# trainer.push_to_hub(f"Nilum/{model_name}")

In [ ]:
sinhala_question = "කේතීකරණ ව්‍යාපෘතියක් සමඟ ඔබ විසඳූ ගැටලුවක් විස්තර කරන්න." # What is the capital of Sri Lanka?
test_prompt = f"<s>[INST] {sinhala_question} [/INST]"
print(test_prompt)

In [ ]:
input_ids = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**input_ids, max_new_tokens=100)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Original Prompt:")
print(test_prompt)
print("\nModel Response:")
print(response)